# Customer NPS Prediction for a Telecom Operator
### Artefact CI — Junior Data Scientist Take-Home Challenge

**Author:** Taryam William Rodrigue Kabore

This notebook builds a machine-learning system that predicts a customer's **NPS category** (Detractor / Passive / Promoter) from their account and service data, so a retention team can prioritise likely detractors among the ~85% of customers who never answer the NPS survey.

The notebook runs top-to-bottom from the raw IBM Cognos Excel files to a saved, deployable model.

**A note on tools:** I used an LLM as a pair-programming assistant to help scaffold code, debug, and structure this notebook. All modelling decisions — target construction, leakage handling, feature choices, and the interpretation of results — are my own, and I have verified every output.

## 1. Data source and assembly

The brief specifies the **IBM Cognos Telco Customer Churn (11.1.3+)** dataset, which — unlike the widely-circulated 21-column Kaggle version — contains a real **Satisfaction Score (1–5)** provided by customers. This score is the basis for the NPS target (Section 4.1 of the brief).

*(Note: I initially used the Kaggle `blastchar` version, which lacks the Satisfaction Score. On re-reading the brief I switched to the Cognos version and rebuilt the target from the real satisfaction signal.)*

The data ships as several tables. I need two:
- **`Telco_customer_churn.xlsx`** — customer features (demographics, services, charges)
- **`Telco_customer_churn_status.xlsx`** — contains the **Satisfaction Score**

I join only the Satisfaction Score onto the feature table. I deliberately do **not** pull the other status columns (`Churn Score`, `Churn Reason`, `Customer Status`, etc.) because they are post-outcome fields that would leak the answer — more on this in Section 2.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the feature table and the status table
main   = pd.read_excel('data/Telco_customer_churn.xlsx')
status = pd.read_excel('data/Telco_customer_churn_status.xlsx')

print('Feature table:', main.shape)
print('Status table :', status.shape)

# Pull ONLY the Satisfaction Score from status (avoid dragging in leaky churn columns)
sat = status[['Customer ID', 'Satisfaction Score']].rename(columns={'Customer ID': 'CustomerID'})

# Join on customer id
df = main.merge(sat, on='CustomerID', how='inner')
print('After merge  :', df.shape)
print('Missing satisfaction after merge:', df['Satisfaction Score'].isna().sum())

Feature table: (7043, 33)
Status table : (7043, 11)
After merge  : (7043, 34)
Missing satisfaction after merge: 0


## 2. Building the NPS target (and avoiding leakage)

### 2.1 The target

NPS groups customers into three ordered classes. I map the Satisfaction Score to NPS using the brief's baseline mapping:

| Satisfaction | NPS class |
|---|---|
| 5 | Promoter |
| 4 | Passive |
| ≤ 3 | Detractor |

This uses a **real, human-provided satisfaction signal** rather than a fabricated proxy.

In [2]:
def to_nps(score):
    if score == 5:   return 'Promoter'
    elif score == 4: return 'Passive'
    else:            return 'Detractor'   # 1, 2, 3

df['NPS'] = df['Satisfaction Score'].apply(to_nps)

dist = df['NPS'].value_counts()
total = len(df)
print('NPS class distribution:')
for c in ['Detractor', 'Passive', 'Promoter']:
    print(f'  {c:10s}: {dist[c]:5d}  ({dist[c]/total*100:4.1f}%)')

net_nps = (dist['Promoter'] - dist['Detractor']) / total * 100
print(f'\nNet NPS score: {net_nps:+.0f}')

NPS class distribution:
  Detractor :  4105  (58.3%)
  Passive   :  1789  (25.4%)
  Promoter  :  1149  (16.3%)

Net NPS score: -42


The result is a realistically **imbalanced, Detractor-heavy** target (58 / 25 / 16). This matters: NPS distributions are always skewed, and the Passive (middle) class is the hardest to separate. I handle the imbalance explicitly during modelling.

### 2.2 Leakage — the critical decision

The business goal is to score customers who have **not** answered the survey and have **not** yet churned. So any column that only becomes known *after* the survey or *after* churn must be excluded from the features — otherwise the model "cheats" using information it would not have at prediction time, and its metrics become meaningless in deployment.

**Dropped — leakage:**
- `Satisfaction Score` — this *is* the target.
- `Churn Label`, `Churn Value`, `Customer Status` — the churn outcome.
- `Churn Score` — a churn *prediction* from IBM SPSS Modeler (using another model's output as a feature).
- `Churn Reason`, `Churn Category` — only exist after a customer leaves.
- `CLTV` — a model-derived lifetime value that likely encodes churn risk. Borderline; I drop it to be safe. (A defensible alternative is to keep it and monitor its SHAP contribution.)

**Dropped — no information:** `CustomerID`, `Count` (always 1), `Country`/`State` (constant — all California, USA), `Lat Long` (text duplicate of numeric lat/long).

**Dropped — fairness:** `City`, `Zip Code`, `Latitude`, `Longitude`. Geography is a proxy for socio-economic status (and often race/income). Using it to allocate retention budget risks uneven treatment across groups (Section 4.7). I drop it deliberately, accepting a small likely accuracy cost in exchange for a cleaner fairness position.

In [3]:
drop_cols = [
    # leakage
    'Satisfaction Score', 'Churn Label', 'Churn Value', 'Churn Score',
    'CLTV', 'Churn Reason',
    # no information
    'CustomerID', 'Count', 'Country', 'State', 'Lat Long',
    # fairness (geographic proxies)
    'City', 'Zip Code', 'Latitude', 'Longitude',
]
# only drop those present (some live only in the status table)
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print('Remaining columns:', df.shape[1])
print(list(df.columns))

Remaining columns: 20
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'NPS']


## 3. Preprocessing

Three things to handle:

1. **`Total Charges` is stored as text** with blank values for 11 brand-new customers (tenure = 0). Since a 0-month customer genuinely has $0 in total charges, I fill those blanks with 0 rather than dropping the rows or imputing a mean.
2. **16 categorical features** need encoding (models can't read "Month-to-month"). I one-hot encode them.
3. **3 numeric features** (tenure, monthly, total charges) are on different scales, so I standardise them.

I use a **stratified** train/test split so both sets preserve the 58/25/16 class balance — important because the minority Promoter class is small and a random split could under-represent it in the test set.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Fix Total Charges: text -> numeric; blanks (new customers) -> 0
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').fillna(0)

y = df['NPS']
X = df.drop(columns=['NPS'])

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges']
print(f'{len(cat_cols)} categorical, {len(num_cols)} numeric features')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
X_train_p = preprocessor.fit_transform(X_train)
X_test_p  = preprocessor.transform(X_test)

16 categorical, 3 numeric features
Train: (5634, 19)  Test: (1409, 19)


## 4. Baseline model — Logistic Regression

The brief is explicit: *start from a sensible baseline before any advanced modelling — it disciplines the rest of the work.* A baseline tells me the floor; I only trust a complex model if it beats this.

I use `class_weight='balanced'` so the model doesn't simply predict "Detractor" for everyone (which would score deceptively on accuracy while being useless for finding the minority classes).

I evaluate with **macro-F1** and **per-class recall**, not accuracy. Accuracy is misleading on an imbalanced target; macro-F1 weights all three classes equally, and per-class recall tells me whether I actually catch Detractors — the class the retention team cares about.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

le = LabelEncoder()
y_train_e = le.fit_transform(y_train)
y_test_e  = le.transform(y_test)

baseline = LogisticRegression(max_iter=1000, class_weight='balanced')
baseline.fit(X_train_p, y_train_e)
base_pred = baseline.predict(X_test_p)

print('BASELINE — Logistic Regression')
print(f'Accuracy : {accuracy_score(y_test_e, base_pred):.3f}')
print(f'Macro F1 : {f1_score(y_test_e, base_pred, average="macro"):.3f}\n')
print(classification_report(y_test_e, base_pred, target_names=le.classes_, digits=3))

BASELINE — Logistic Regression
Accuracy : 0.462
Macro F1 : 0.414

              precision    recall  f1-score   support

   Detractor      0.742     0.493     0.593       821
     Passive      0.292     0.204     0.240       358
    Promoter      0.282     0.752     0.410       230

    accuracy                          0.462      1409
   macro avg      0.439     0.483     0.414      1409
weighted avg      0.552     0.462     0.473      1409



## 5. Main model — XGBoost

Gradient boosting is a strong default for tabular data. I handle the imbalance with balanced sample weights (the XGBoost equivalent of `class_weight`), and keep the model moderately regularised (`max_depth=4`, `learning_rate=0.05`) to avoid overfitting a weak signal.

In [6]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight('balanced', y_train_e)

xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    objective='multi:softprob', num_class=3,
    eval_metric='mlogloss', random_state=42
)
xgb.fit(X_train_p, y_train_e, sample_weight=sample_weights)
xgb_pred = xgb.predict(X_test_p)

print('MAIN MODEL — XGBoost')
print(f'Accuracy : {accuracy_score(y_test_e, xgb_pred):.3f}')
print(f'Macro F1 : {f1_score(y_test_e, xgb_pred, average="macro"):.3f}\n')
print(classification_report(y_test_e, xgb_pred, target_names=le.classes_, digits=3))

MAIN MODEL — XGBoost
Accuracy : 0.463
Macro F1 : 0.413

              precision    recall  f1-score   support

   Detractor      0.724     0.525     0.609       821
     Passive      0.283     0.257     0.269       358
    Promoter      0.266     0.565     0.362       230

    accuracy                          0.463      1409
   macro avg      0.424     0.449     0.413      1409
weighted avg      0.537     0.463     0.482      1409



### Interpreting the result — an honest finding

XGBoost (macro-F1 ≈ 0.41) **matches but does not beat** the logistic baseline (≈ 0.41). This is a genuine, informative result, not a failure:

- When a simple linear model and a tuned gradient-boosting model land in the same place, the limiting factor is the **features, not the model.** Account and service data carry only weak signal about *felt* satisfaction — how satisfied someone is depends heavily on experiences not captured in tabular data (a bad support call, expectations, mood).
- This is exactly why the brief encourages adding **customer verbatims** (Section 4.4): free-text is where the real satisfaction signal lives.

I also note that my earlier (incorrect) approach — deriving the target from churn/tenure and then using churn-correlated features — produced an *inflated* macro-F1 around 0.74. That number was a **leakage artefact**. The honest, leak-free number is ~0.41, and it is the one I trust. The brief values this honesty over polish.

For a retention team, the most business-relevant metric is **Detractor recall** — how many at-risk customers we catch. The model catches ~53% of Detractors at ~72% precision.

## 6. Fairness audit

A model that allocates retention budget by predicted NPS can quietly treat demographic groups unequally. I audit **Detractor recall across groups** — does the model catch unhappy customers equally regardless of demographics?

In [7]:
det_idx = list(le.classes_).index('Detractor')
X_test_r = X_test.reset_index(drop=True)
y_test_r = pd.Series(y_test_e).reset_index(drop=True)
pred_r   = pd.Series(xgb_pred)

for attr in ['Gender', 'Senior Citizen', 'Partner', 'Dependents']:
    print(f'Detractor recall by {attr}:')
    for val in sorted(X_test_r[attr].unique()):
        mask = X_test_r[attr] == val
        is_det = (y_test_r[mask] == det_idx)
        if is_det.sum() > 0:
            recall = ((pred_r[mask] == det_idx) & is_det).sum() / is_det.sum()
            print(f'  {val:8s}: {recall:.3f}  (n={is_det.sum()})')
    print()

Detractor recall by Gender:
  Female  : 0.528  (n=396)
  Male    : 0.522  (n=425)

Detractor recall by Senior Citizen:
  No      : 0.488  (n=678)
  Yes     : 0.699  (n=143)

Detractor recall by Partner:
  No      : 0.610  (n=433)
  Yes     : 0.430  (n=388)

Detractor recall by Dependents:
  No      : 0.597  (n=655)
  Yes     : 0.241  (n=166)



**Findings:**
- **Gender:** fair (≈0.52 both).
- **Senior Citizen:** the model catches unhappy seniors *better* than non-seniors.
- **Dependents:** a serious disparity — it catches ~60% of unhappy customers **without** dependents but only ~24% **with** dependents.

The Dependents gap is the kind of finding I would **escalate to Customer Experience / Legal before production** (Section 4.7). Using the model as-is would systematically under-serve unhappy customers with dependents. The likely cause is limited signal for that subgroup combined with class imbalance; a mitigation would be subgroup-aware thresholds or targeted data collection.

## 7. Drivers of detraction (interpretability)

I use SHAP to identify which features drive the predictions, and separate **actionable** levers (the business can change them) from **non-actionable** ones.

In [8]:
import shap

feat_names = num_cols + list(
    preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
)

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_p)

if isinstance(shap_values, list):
    importance = np.mean([np.abs(s).mean(0) for s in shap_values], axis=0)
else:
    importance = np.abs(shap_values).mean(0) if shap_values.ndim == 2 \
                 else np.abs(shap_values).mean(axis=(0, 2))

imp_df = (pd.DataFrame({'feature': feat_names, 'importance': importance})
          .sort_values('importance', ascending=False).head(12))
print('Top 12 drivers (SHAP global importance):')
for _, r in imp_df.iterrows():
    print(f'  {r.feature:35s} {r.importance:.4f}')

Top 12 drivers (SHAP global importance):
  Online Security_Yes                 0.3717
  Monthly Charges                     0.1354
  Tenure Months                       0.1085
  Total Charges                       0.1027
  Online Security_No                  0.0655
  Contract_Month-to-month             0.0565
  Dependents_No                       0.0494
  Internet Service_Fiber optic        0.0435
  Tech Support_No                     0.0396
  Paperless Billing_No                0.0367
  Internet Service_No                 0.0358
  Partner_No                          0.0265


**Reading the drivers:**
- **Online Security** is the strongest signal, followed by **Monthly Charges**, **Tenure**, and **Contract type**.
- **Actionable levers:** Online Security, Tech Support, Contract type — the company can offer these. For a predicted Detractor, the single most likely lever is **encouraging adoption of support/security add-ons and moving them off a month-to-month contract.**
- **Less actionable:** tenure and charges.

**Correlation vs causation:** Online Security dominating may reflect that security-subscribers are simply more *engaged* customers, rather than the add-on *causing* satisfaction. I treat these drivers as correlational, not causal.

## 8. Persisting the model

I save the trained model, the fitted preprocessor, and the label encoder so the prediction app can load them and score new customers without retraining.

In [9]:
import joblib

joblib.dump(xgb, 'models/nps_xgb_model.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')
joblib.dump(le, 'models/label_encoder.pkl')
print('Saved: model, preprocessor, label encoder')

Saved: model, preprocessor, label encoder


## 9. Limitations and next steps

**Limitations**
- **Weak tabular ceiling.** Account data explains satisfaction only weakly (macro-F1 ≈ 0.41). The Passive class in particular is hard to separate.
- **Derived target.** NPS is mapped from a 1–5 satisfaction score, not true 0–10 NPS survey responses, which introduces some noise at the class boundaries.
- **Fairness gap.** The Dependents subgroup disparity needs mitigation before any production use.
- **Geography dropped.** I traded some accuracy for a cleaner fairness position; this is a defensible but reversible choice.

**Next steps**
- **Add customer verbatims (Section 4.4):** generate satisfaction-conditioned free-text notes and add sentiment/embedding features — the most promising route past the tabular ceiling.
- **Threshold tuning** toward Detractor recall for the retention use-case.
- **Subgroup-aware calibration** to close the Dependents fairness gap.
- **Monitoring** for input/prediction drift once real survey responses arrive.